In [0]:
from pyspark.sql.functions import col, max as spark_max, datediff, lit, when
from pyspark.sql.types import IntegerType

storage_account= "stfintechpipeline"
storage_account_key = "YOUR_ACCESS_KEY_HERE"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_account_key
)

CURATED = f"abfss://curated@{storage_account}.dfs.core.windows.net"

transaction_features  = spark.read.parquet(f"{CURATED}/transaction_features/")

print(f"Rows    : {transaction_features .count():,}")
print(f"Columns : {len(transaction_features.columns)}")
transaction_features.printSchema()

Rows    : 13,305,915
Columns : 19
root
 |-- id: long (nullable = true)
 |-- client_id: integer (nullable = true)
 |-- amount_clean: double (nullable = true)
 |-- transaction_hour: integer (nullable = true)
 |-- is_weekend: boolean (nullable = true)
 |-- use_chip_encoded: integer (nullable = true)
 |-- merchant_state: string (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- error_encoded: integer (nullable = true)
 |-- is_fraud_label: integer (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- debt_to_income_ratio: double (nullable = true)
 |-- avg_spend_per_transaction: double (nullable = true)
 |-- error_rate: double (nullable = true)
 |-- pct_online_transactions: double (nullable = true)
 |-- max_account_tenure_days: integer (nullable = true)
 |-- amount_vs_customer_avg: double (nullable = true)
 |-- is_home_state: integer (nullable = true)
 |-- is_high_risk_hour: integer (nullable = true)



In [0]:
transaction_features_fixed = spark.read.parquet(f"{CURATED}/transaction_features/")

print("=== Fraud Label Distribution ===")
transaction_features_fixed.groupBy("is_fraud_label") \
    .count() \
    .orderBy("is_fraud_label") \
    .show()

=== Fraud Label Distribution ===
+--------------+--------+
|is_fraud_label|   count|
+--------------+--------+
|             0|13292583|
|             1|   13332|
+--------------+--------+



is_fraud_label is ground truth from train_fraud_labels.json
  0 = legitimate transaction
  1 = confirmed fraud transaction

#Section 1- feature preparation
target encoding for categorical columns
class weighting for imbalanced fraud labels

In [0]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.sql.functions import col, when, lit, count, sum as spark_sum
from pyspark.sql.types import IntegerType, DoubleType

#feature preparation
#load transacton features
df_fraud = spark.read.parquet(f"{CURATED}/transaction_features/")

#cast boolean to integer
df_fraud=df_fraud.withColumn("is_weekend",
                             col("is_weekend").cast(IntegerType()))
#target encoding for merchant state by replacing each state with its fraud rate
state_fraud_rate = df_fraud.groupBy("merchant_state").agg(
    spark_sum("is_fraud_label").alias("fraud_count"),
    count("*").alias("total_count")
).withColumn("merchant_state_encoded",
             (col("fraud_count")/col("total_count")).cast(DoubleType())
).select("merchant_state", "merchant_state_encoded")

df_fraud = df_fraud.join(state_fraud_rate, on="merchant_state",how="left")
#target encoding for merchant category by replacing each category with its fraud rate
category_fraud_rate = df_fraud.groupBy("merchant_category").agg(
    spark_sum("is_fraud_label").alias("fraud_count"),
    count("*").alias("total_count")
).withColumn("merchant_category_encoded",
             (col("fraud_count")/col("total_count")).cast(DoubleType())
).select("merchant_category", "merchant_category_encoded")

df_fraud = df_fraud.join(category_fraud_rate, on="merchant_category",how="left")

#define final feature list
fraud_features =[
    "amount_clean", "transaction_hour", "is_weekend",
    "use_chip_encoded", "error_encoded",
    "credit_score", "debt_to_income_ratio",
    "avg_spend_per_transaction", "error_rate",
    "pct_online_transactions", "max_account_tenure_days",
    "amount_vs_customer_avg", "is_home_state",
    "is_high_risk_hour", "merchant_state_encoded",
    "merchant_category_encoded" 
]

# fill any nulls with 0
df_fraud = df_fraud.fillna(0, subset=fraud_features)

#assemble feature vector
assembler = VectorAssembler(
    inputCols=fraud_features,
    outputCol="features_raw"
)
df_fraud=assembler.transform(df_fraud)

#scale features
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features_scaled",
)
scaler_model =scaler.fit(df_fraud)
df_fraud=scaler_model.transform(df_fraud)


print(f"Features: {len(fraud_features)}")
print(f"Total rows: {df_fraud.count():,}")
print("Feature preparation complete")

Features: 16
Total rows: 13,305,915
Feature preparation complete


did target encoding for merchant_state and merchant_category as fraud rates since it captures geographic/merchant signal without add 100+ one-hot columns
is_weekend cast to integer since boolean is not accepted in MLib
StandardScaler is applied to Logistic Regression only
Random Forest uses raw unscaled features

In [0]:
from pyspark.sql.functions import col

#train/test split and class weights

#train/test split
train_df, test_df = df_fraud.randomSplit([0.8,0.2], seed=42)
print(f"Training rows : {train_df.count():,}")
print(f"Test rows     : {test_df.count():,}")

#calculate class weights
#weight=total rows / (num classes * class count). this tells the model missing fraud is more costly
total = train_df.count()
fraud_count = train_df.filter(col("is_fraud_label")== 1).count()
non_fraud_count = train_df.filter(col("is_fraud_label")==0).count()

weight_fraud=total/(2*fraud_count)
weight_non_fraud= total/(2*non_fraud_count)

print(f"\nFraud transactions: {fraud_count:,}")
print(f"Non-fraud transactions: {non_fraud_count:,}")
print(f"Weight for fraud (1) : {weight_fraud:.2f}")
print(f"Weight for non-fraud (0): {weight_non_fraud:.2f}")

#add weight column to training data
train_df=train_df.withColumn("class_weight",
                             when(col("is_fraud_label")==1,weight_fraud)
                             .otherwise(weight_non_fraud))
print("\nClass weights applied")

Training rows : 10,642,742
Test rows     : 2,663,173

Fraud transactions: 10,726
Non-fraud transactions: 10,632,016
Weight for fraud (1) : 496.12
Weight for non-fraud (0): 0.50

Class weights applied


fraud rate: fraud_cnt/total*100:,2f% is severe imbalance
a model predicting all transactions as legitimate achieves 99.9% accuracy but catches 0 fraud which is useless for a bank
wieght ratios(weight_fraud/weigh_legit)- 1 penalises missing fraud proportionally to its rarity, forcing the model to prioritise recall
SMOTE was considered but pyspark has no native implementation and converting 13M rows to pandas for imblearn is computationally infeasible

#Section 2- supevised models

In [0]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)

#Logistic Regression with Class Weighting
lr= LogisticRegression(
    featuresCol="features_scaled",
    labelCol="is_fraud_label",
    weightCol="class_weight",
    maxIter=100,
    regParam=0.01,
    elasticNetParam=0.0
)

print("Training Logistic Regression...")
lr_model = lr.fit(train_df)
lr_predictions = lr_model.transform(test_df)

Training Logistic Regression...


LR Parameters:
  maxIter=100 : iterations for optimisation convergence
  regParam=0.01: small L2 regularisation — prevents overfitting without aggressively shrinking coefficients
  elasticNetParam=0.0 : pure L2 (Ridge) regularisation
  featuresCol=features_scaled : LR is scale-sensitive,
  StandardScaler applied before training

In [0]:
#evaluate
auc_roc = BinaryClassificationEvaluator(
    labelCol="is_fraud_label",
    metricName="areaUnderROC"
).evaluate(lr_predictions)
auc_pr = BinaryClassificationEvaluator(
    labelCol="is_fraud_label",
    metricName="areaUnderPR"
).evaluate(lr_predictions)

accuracy= MulticlassClassificationEvaluator(
    labelCol="is_fraud_label",
    metricName="accuracy"
).evaluate(lr_predictions)

precision= MulticlassClassificationEvaluator(
    labelCol="is_fraud_label",
    metricName="weightedPrecision"
).evaluate(lr_predictions)

recall = MulticlassClassificationEvaluator(
    labelCol="is_fraud_label",
    metricName="weightedRecall"
).evaluate(lr_predictions)

print("\nLogistic Regression Results")
print(f"AUC-ROC   : {auc_roc:.4f}")
print(f"AUC-PR    : {auc_pr:.4f}")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")


Logistic Regression Results
AUC-ROC   : 0.9522
AUC-PR    : 0.1099
Accuracy  : 0.8736
Precision : 0.9989
Recall    : 0.8736


In [0]:
from pyspark.sql.functions import col

# Fraud-specific precision and recall
tp = lr_predictions.filter((col("prediction") == 1) & (col("is_fraud_label") == 1)).count()
fp = lr_predictions.filter((col("prediction") == 1) & (col("is_fraud_label") == 0)).count()
fn = lr_predictions.filter((col("prediction") == 0) & (col("is_fraud_label") == 1)).count()
tn = lr_predictions.filter((col("prediction") == 0) & (col("is_fraud_label") == 0)).count()

fraud_precision = tp / (tp + fp) if (tp + fp) > 0 else 0
fraud_recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * (fraud_precision * fraud_recall) / (fraud_precision + fraud_recall) if (fraud_precision + fraud_recall) > 0 else 0

print("=== LR Fraud-Specific Metrics ===")
print(f"True Positives  (caught fraud)    : {tp:,}")
print(f"False Positives (false alarms)    : {fp:,}")
print(f"False Negatives (missed fraud)    : {fn:,}")
print(f"True Negatives  (correct non-fraud): {tn:,}")
print(f"\nFraud Precision : {fraud_precision:.4f}")
print(f"Fraud Recall    : {fraud_recall:.4f}")
print(f"Fraud F1 Score  : {f1:.4f}")

=== LR Fraud-Specific Metrics ===
True Positives  (caught fraud)    : 2,399
False Positives (false alarms)    : 336,311
False Negatives (missed fraud)    : 207
True Negatives  (correct non-fraud): 2,324,256

Fraud Precision : 0.0071
Fraud Recall    : 0.9206
Fraud F1 Score  : 0.0141


In [0]:
from pyspark.ml.classification import RandomForestClassifier

#random forest with class weighting
rf= RandomForestClassifier(
    featuresCol="features_scaled",
    labelCol="is_fraud_label",
    weightCol="class_weight",
    numTrees=100,
    maxDepth=10,
    seed=42
)

print("Training Random Forest...")
rf_model = rf.fit(train_df)
rf_predictions = rf_model.transform(test_df)

Training Random Forest...


RF Parameters:
  numTrees=100  : 100 decision trees averaged — reduces variance vs single tree, industry standard starting point
  maxDepth=10: allows complex multi-condition fraud patterns e.g. online + night + high amount + new merchant
  featuresCol=features_raw : RF is scale-invariant,
  no StandardScaler needed
  weightCol=class_weight : same fraud penalty as LR

Note: Parameter tuning via CrossValidator was attempted but computationally infeasible on 13M rows with student cluster(estimated 11+ hours). Future work: Databricks AutoML or MLflow hyperparameter optimisation.

In [0]:
#evaluate
rf_auc_roc = BinaryClassificationEvaluator(
    labelCol="is_fraud_label",
    metricName="areaUnderROC"
).evaluate(rf_predictions)

rf_auc_pr = BinaryClassificationEvaluator(
    labelCol="is_fraud_label",
    metricName="areaUnderPR"
).evaluate(rf_predictions)

rf_accuracy = MulticlassClassificationEvaluator(
    labelCol="is_fraud_label",
    metricName="accuracy"
).evaluate(rf_predictions)

rf_precision = MulticlassClassificationEvaluator(
    labelCol="is_fraud_label",
    metricName="weightedPrecision"
).evaluate(rf_predictions)

rf_recall = MulticlassClassificationEvaluator(
    labelCol="is_fraud_label",
    metricName="weightedRecall"
).evaluate(rf_predictions)

print("\n=== Random Forest Results ===")
print(f"AUC-ROC   : {rf_auc_roc:.4f}")
print(f"AUC-PR    : {rf_auc_pr:.4f}")
print(f"Accuracy  : {rf_accuracy:.4f}")
print(f"Precision : {rf_precision:.4f}")
print(f"Recall    : {rf_recall:.4f}")


=== Random Forest Results ===
AUC-ROC   : 0.9843
AUC-PR    : 0.1887
Accuracy  : 0.9684
Precision : 0.9990
Recall    : 0.9684


In [0]:
# RF fraud specific metrics
tp_rf = rf_predictions.filter((col("prediction") == 1) & (col("is_fraud_label") == 1)).count()
fp_rf = rf_predictions.filter((col("prediction") == 1) & (col("is_fraud_label") == 0)).count()
fn_rf = rf_predictions.filter((col("prediction") == 0) & (col("is_fraud_label") == 1)).count()
tn_rf = rf_predictions.filter((col("prediction") == 0) & (col("is_fraud_label") == 0)).count()

fraud_precision_rf = tp_rf / (tp_rf + fp_rf) if (tp_rf + fp_rf) > 0 else 0
fraud_recall_rf = tp_rf / (tp_rf + fn_rf) if (tp_rf + fn_rf) > 0 else 0
f1_rf = 2 * (fraud_precision_rf * fraud_recall_rf) / (fraud_precision_rf + fraud_recall_rf) if (fraud_precision_rf + fraud_recall_rf) > 0 else 0

print("=== RF Fraud-Specific Metrics ===")
print(f"True Positives  (caught fraud)     : {tp_rf:,}")
print(f"False Positives (false alarms)     : {fp_rf:,}")
print(f"False Negatives (missed fraud)     : {fn_rf:,}")
print(f"True Negatives  (correct non-fraud): {tn_rf:,}")
print(f"\nFraud Precision : {fraud_precision_rf:.4f}")
print(f"Fraud Recall    : {fraud_recall_rf:.4f}")
print(f"Fraud F1 Score  : {f1_rf:.4f}")

=== RF Fraud-Specific Metrics ===
True Positives  (caught fraud)     : 2,433
False Positives (false alarms)     : 83,962
False Negatives (missed fraud)     : 173
True Negatives  (correct non-fraud): 2,576,605

Fraud Precision : 0.0282
Fraud Recall    : 0.9336
Fraud F1 Score  : 0.0547


In [0]:
#feature importance
print("top 10 features")
importances = rf_model.featureImportances
feature_importance=[(fraud_features[i], float(importances[i]))for i in range(len(fraud_features))]
feature_importance_sorted = sorted(feature_importance, key=lambda x: x[1], reverse=True)
for feat, imp in feature_importance_sorted[:10]:
    print(f"{feat} : {imp:.4f}")

top 10 features
merchant_state_encoded : 0.3852
is_home_state : 0.2427
merchant_category_encoded : 0.2146
use_chip_encoded : 0.0448
amount_vs_customer_avg : 0.0320
transaction_hour : 0.0213
pct_online_transactions : 0.0213
amount_clean : 0.0194
avg_spend_per_transaction : 0.0040
error_rate : 0.0033


In [0]:
# Full comparison LR vs RF
print("\n=== Complete Model Comparison ===")
print(f"{'Metric':<30} {'LR':>15} {'RF':>15}")
print("-" * 60)
print(f"{'AUC-ROC':<30} {0.9522:>15.4f} {rf_auc_roc:>15.4f}")
print(f"{'AUC-PR':<30} {0.1099:>15.4f} {rf_auc_pr:>15.4f}")
print(f"{'Fraud Recall':<30} {0.9206:>15.4f} {fraud_recall_rf:>15.4f}")
print(f"{'Fraud Precision':<30} {0.0071:>15.4f} {fraud_precision_rf:>15.4f}")
print(f"{'False Alarms':<30} {336311:>15,} {fp_rf:>15,}")
print(f"{'Fraud F1':<30} {0.0141:>15.4f} {f1_rf:>15.4f}")


=== Complete Model Comparison ===
Metric                                      LR              RF
------------------------------------------------------------
AUC-ROC                                 0.9522          0.9843
AUC-PR                                  0.1099          0.1887
Fraud Recall                            0.9206          0.9336
Fraud Precision                         0.0071          0.0282
False Alarms                           336,311          83,962
Fraud F1                                0.0141          0.0547


In [0]:
#save RF model
rf_model.save(f"{CURATED}/models/rf_fraud_model")
print("RF model saved to curated/ ✓")

# Verify it saved correctly
print("Model saved at:")
print(f"{CURATED}/models/rf_fraud_model")

RF model saved to curated/ ✓
Model saved at:
abfss://curated@stfintechpipeline.dfs.core.windows.net/models/rf_fraud_model


Random forest selected as primary model
  - higher AUC-PR(better fraud class classification)
  - higher fraud recall(catches more real fraud)
  - 75% fewer flase alarms than logistic regression
  - built-in feature importance for interpretability
  - location features dominate for identifying fraud- merchant_state_encoded(38%), is_home_state(24%), merchant_category_encoded(21%)
  - fraud is primarily a geographic and merchant type occurence

#Section 3- unsupervised anomaly detection
isolation forest- finds fraud without labels

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import col, when
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    roc_auc_score, 
    average_precision_score,
    classification_report
)
import numpy as np

#isolation forest- unsupervised anomaly detection
#since pyspark MLib doesnt have isolation forest we sample the data and use scikit-learn

#sample data for isolation forest
#all 13,332 real fraud cases preserved in sample
fraud_sample = df_fraud.filter(col("is_fraud_label")==1)
non_fraud_sample = df_fraud.filter(col("is_fraud_label")==0).sample(fraction=0.01, seed=42)
df_if_sample = fraud_sample.union(non_fraud_sample)

print(f"Isolation Forest sample size: {df_if_sample.count():,}")
print(f"Fraud in sample: {fraud_sample.count():,}")
print(f"Non-fraud in sample: {non_fraud_sample.count():,}")

#convert to pandas
pdf = df_if_sample.select(fraud_features + ["is_fraud_label"])\
    .fillna(0)\
    .toPandas()

X=pdf[fraud_features].values
y=pdf["is_fraud_label"].values

#train isolation forest
#contamination =expected proportion of fraud in dataset
contamination = 13332/13305915

print(f"\nContamination rate: {contamination:.4f}")
print("Training Isolation Forest...")

iso_forest = IsolationForest(
    n_estimators=100,
    contamination=contamination,
    random_state=42,
    n_jobs=1
)
iso_forest.fit(X)

Isolation Forest sample size: 146,968
Fraud in sample: 13,332
Non-fraud in sample: 133,636

Contamination rate: 0.0010
Training Isolation Forest...


IsolationForest(contamination=0.001001960406330568, n_jobs=1, random_state=42)

In [0]:
#generate anomaly scores
#raw score-more negative=more anomalous=more likely fraud
anomaly_scores = iso_forest.decision_function(X)
predictions_if= iso_forest.predict(X)

# Convert Isolation Forest output to match fraud labels
# Isolation Forest: -1 = anomaly (fraud), 1 = normal (not fraud)
# Convert to: 1 = fraud, 0 = not fraud
y_pred_binary = np.where(predictions_if== -1,1,0)

#convert anomaly scores to probability-like scores
#more negative score=higher fraud probability
scores_normalised = 1-(anomaly_scores-anomaly_scores.min())/\
    (anomaly_scores.max()-anomaly_scores.min())

In [0]:
#evaluate
auc_roc_if = roc_auc_score(y, scores_normalised)
auc_pr_if = average_precision_score(y,scores_normalised)

print("\n=== Isolation Forest Results ===")
print(f"AUC-ROC : {auc_roc_if:.4f}")
print(f"AUC-PR  : {auc_pr_if:.4f}")
print("\nClassification Report:")
print(classification_report(y, y_pred_binary, 
      target_names=["Not Fraud", "Fraud"]))



=== Isolation Forest Results ===
AUC-ROC : 0.8627
AUC-PR  : 0.4020

Classification Report:
              precision    recall  f1-score   support

   Not Fraud       0.91      1.00      0.95    133492
       Fraud       0.80      0.01      0.02     13332

    accuracy                           0.91    146824
   macro avg       0.85      0.50      0.49    146824
weighted avg       0.90      0.91      0.87    146824



In [0]:
#compare what isolaion forest found vs actual labels
if_results = pd.DataFrame({
    "actual":y,
    "predicted": y_pred_binary,
    "anomaly_score": scores_normalised
})

print("\nIsolation Forest Confusion")
print(pd.crosstab(if_results["actual"], 
                  if_results["predicted"],
                  rownames=["Actual"], 
                  colnames=["Predicted"]))


Isolation Forest Confusion
Predicted       0    1
Actual                
0          133462   30
1           13214  118


AUC-PR OF 0.4- highest of all 3 models
fraud recall 1%- binary threshold extremely conservative
the anomaly score ranks fraud well(high AUC-PR) but the binary decision flags very few transactions. When isolation forest flags a transaction, it is almost certainly fraud(near 0 false alarms)
this makes it ideal for a high-confidence autoblock tier- combined with RF in the layered detection system

#Section 4- layered fraud detection system
combined RF probability + IF score

In [0]:
#layered fraud detection
from pyspark.ml.functions import vector_to_array
import pandas as pd
import numpy as np

#combines RF probability + IF anomaly score
#get RF probability score for all transactions
print("generating RF scores for full dataset")
all_rf_predictions = rf_model.transform(df_fraud)
all_rf_predictions = all_rf_predictions\
    .withColumn("rf_fraud_probability",
                vector_to_array(col("probability"))[1])
    
#get IF anomaly scores
print("generating IF scores for full dataset")
pdf_full = df_fraud.select(
    fraud_features + ["id", "client_id", "is_fraud_label"]
).fillna(0).toPandas()

X_full = pdf_full[fraud_features].values
iso_forest_final = IsolationForest(
    n_estimators = 100,
    contamination=13332/13305915,
    random_state=42,
    n_jobs=1
)
iso_forest_final.fit(X_full)

raw_scores = iso_forest_final.decision_function(X_full)

#normalize IF scores to 0-1
if_scores_normalised = 1 - (raw_scores - raw_scores.min()) / \
    (raw_scores.max() - raw_scores.min())

pdf_full["if_anomaly_score"] = if_scores_normalised

#convert IF scores back to Spark
if_scores_spark = spark.createDataFrame(
    pdf_full[["id","if_anomaly_score"]]
)

generating RF scores for full dataset
generating IF scores for full dataset


In [0]:
from pyspark.sql.functions import round as spark_round
#join RF and IF scores
combined_df = all_rf_predictions\
    .select("id","client_id", "is_fraud_label", "rf_fraud_probability")\
    .join(if_scores_spark, on="id", how="left")

#calculate combined score
#in this case RF gets 60% weight because it has trained on labels while IF gets 40% weight since its unsupervised signal
combined_df = combined_df.withColumn(
    "combined_fraud_score",
    spark_round(
        (col("rf_fraud_probability")*0.6)+
        (col("if_anomaly_score")*0.4),4
    ) 
)


In [0]:
#assign fraud tiers
combined_df = combined_df.withColumn(
    "fraud_tier",
    when(col("combined_fraud_score")>0.8, "Tier 1-Auto Block")
    .when(col("combined_fraud_score")>0.5, "Tier 2-Investigate")
    .otherwise("Tier 3-Monitor")
)

In [0]:
# Evaluate tier distribution
print("\n=== Fraud Tier Distribution ===")
combined_df.groupBy("fraud_tier").count() \
    .orderBy("fraud_tier").show()

#Check how much real fraud each tier catches
print("=== Real Fraud Caught Per Tier ===")
combined_df.groupBy("fraud_tier").agg(
    spark_sum(col("is_fraud_label")).alias("fraud_caught"),
    count("*").alias("total_flagged")
).withColumn("fraud_rate",
    spark_round(col("fraud_caught") / col("total_flagged") * 100, 2)
).orderBy("fraud_tier").show()


=== Fraud Tier Distribution ===
+------------------+--------+
|        fraud_tier|   count|
+------------------+--------+
| Tier 1-Auto Block|   56620|
|Tier 2-Investigate|  364497|
|    Tier 3-Monitor|12884798|
+------------------+--------+

=== Real Fraud Caught Per Tier ===
+------------------+------------+-------------+----------+
|        fraud_tier|fraud_caught|total_flagged|fraud_rate|
+------------------+------------+-------------+----------+
| Tier 1-Auto Block|        4923|        56620|      8.69|
|Tier 2-Investigate|        7542|       364497|      2.07|
|    Tier 3-Monitor|         867|     12884798|      0.01|
+------------------+------------+-------------+----------+



Tier 1 — Auto Block
  Both models agree this is high-confidence fraud.
  Blocked immediately — no human review required.
  High fraud density (8.69%) — operationally justified.

Tier 2 — Investigate
  One or both models flag as suspicious.
  Queued for fraud investigator review within 24 hours.
  2.07% fraud rate — manageable investigation volume.

Tier 3 — Monitor
  Both models consider normal. Transaction proceeds.
  0.01% fraud rate — negligible miss rate.

Combined detection rate: ~93.5% of all fraud caught.
Only 6.5% slips through undetected.

In [0]:
#write finalcombined predictions to curated
#write fraud predictions
combined_df.select(
   "id", "client_id",
    "rf_fraud_probability",
    "if_anomaly_score",
    "combined_fraud_score",
    "fraud_tier",
    "is_fraud_label" 
).write.mode("overwrite").parquet(f"{CURATED}/fraud_predictions/")
print("Fraud predictions written")

Fraud predictions written
